# Hybrid RAG, built one stage at a time

### Vector retrieval for prose, graph traversal for structure

A companion notebook (`KG_vs_NaiveRAG_FFCS.ipynb`) measured where each pure approach fails on
`small.pdf`. The result was not that one wins:

| | Naive RAG | Knowledge graph |
|---|---|---|
| Local facts, prose, rationale | correct | **no data — never modelled** |
| Counts and exhaustive lists | **wrong — top-k is a sample** | correct |

They fail in **opposite directions**. That is the whole case for hybrid, and it is a specific,
testable claim rather than a vague "combining is better".

This notebook builds the hybrid pipeline stage by stage, prints the actual prompt it assembles,
and evaluates it — including **four real bugs** found while writing it, all kept in, because
each one teaches something a working example cannot.

---

### The five stages

```
  question
     |
  [1] vector search over Chroma          -> chunk ids + text + citations
     |
  [2] JOIN on chunk id  ------------------> Neo4j (:Chunk {id})
     |
  [3] graph expansion, 1 hop             -> facts spanning chunk boundaries
     |
  [4] type words read from the QUESTION  -> enter the graph independently of retrieval
     |
  [5] census of each relevant type       -> complete sets, for counting
     |
  assemble PASSAGES + GRAPH FACTS + COMPLETE SETS -> LLM -> cited answer
```

Stage 4 is the one most implementations omit. Skipping it turns hybrid back into vector RAG
with extra steps — demonstrated below.

## 0. Setup

Requires the artefacts built by the earlier steps, and Neo4j running
(`docker start ffcs-neo4j`, browser at http://localhost:7474).

In [1]:
# Make the workshop modules importable and data paths resolvable from anywhere.
import sys
from pathlib import Path

SRC = Path.cwd().parent / 'src' if (Path.cwd().parent / 'src').is_dir() else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
import json, textwrap

from pathlib import Path



import ablation, config as C, kag

import hybrid_rag as H

from build_vectorstore import get_collection, get_model, embed_query



chunks = {c['id']: c for c in json.loads(C.CHUNKS_JSON.read_text())['chunks']}

results = json.loads(C.HYBRID_RESULTS.read_text())

hybrid_gaps = {r['id']: r for r in results['hybrid_on_gaps']}

hybrid_agg  = {r['id']: r for r in results['hybrid_on_aggregates']}

naive_gaps  = {r['id']: r for r in results['naive_on_gaps']}

naive       = {r['id']: r for r in json.loads(C.NAIVE_RESULTS.read_text())}



model, collection = get_model(), get_collection()

driver, database = kag.connect()



QUESTION = 'List all the facilities at the school that offers B.Des.'

print(f'Chroma: {collection.count()} vectors   |   worked example: {QUESTION!r}')

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13949.94it/s]

Chroma: 18 vectors   |   worked example: 'List all the facilities at the school that offers B.Des.'


## Stage 1 — Vector retrieval

Ordinary dense retrieval: embed the question, take the nearest chunks. Each chunk carries a
citation (`page N, passage M`) so anything quoted later can be traced back to the document.

Watch what happens for the worked example: retrieval finds the chunk saying V-SIGN offers B.Des,
but **misses the chunk listing the facilities**. On its own, this retrieval cannot answer the
question. Everything that follows is about repairing that.

In [2]:
hit = collection.query(query_embeddings=[embed_query(model, QUESTION)],
                       n_results=4, include=['documents', 'metadatas', 'distances'])
ids = hit['ids'][0]
for cid, meta, dist, doc in zip(ids, hit['metadatas'][0], hit['distances'][0], hit['documents'][0]):
    print(f"  {cid:<14} sim={1-dist:.3f}  ({meta['citation']})")
    print(f"     {doc[:96].strip()}...")
print()
print('chunk_p3_05 (the one naming the five facilities) retrieved?',
      'chunk_p3_05' in ids)

  chunk_p3_06    sim=0.643  (page 3, passage 6 of 6 (paragraph 2))
     . Admission information for B.Des and M.Des programme - Selection for B.Des programme is based o...
  chunk_p3_04    sim=0.600  (page 3, passage 4 of 6 (paragraph 2))
     . V-SIGN, VIT School of Design is one of the newest schools in VIT, Vellore. It has been functio...
  chunk_p3_02    sim=0.593  (page 3, passage 2 of 6 (paragraph 1))
     . Dates of the entrance examinations such as VITEEE, VITMEE will be announced separately through...
  chunk_p1_05    sim=0.577  (page 1, passage 5 of 5)
     5. Students have the option of choosing courses from a ‘basket of courses’ that are grouped into...

chunk_p3_05 (the one naming the five facilities) retrieved? False


## Stage 2 — The join: a chunk id is the bridge between the two stores

This is the mechanism that makes the system hybrid rather than two systems side by side.

```
  Chroma          id = "chunk_p3_04"  ->  embedding, text, citation
                        |
                        |  same string
                        v
  Neo4j    (:Chunk {id: "chunk_p3_04"})  ->  MENTIONED_IN edges to entities
```

Because the loader used the same ids in both stores, a retrieval result is directly a graph entry
point. No fuzzy matching, no embedding of entity names, no second lookup service.

In [3]:
with driver.session(database=database) as s:
    rows = list(s.run('''
        MATCH (e)-[m:MENTIONED_IN]->(c:Chunk) WHERE c.id IN $ids
        RETURN c.id AS chunk, labels(e)[0] AS label,
               coalesce(e.name, e.version, toString(e.number)) AS entity,
               m.surface_form AS matched_text
        ORDER BY chunk, label, entity''', {'ids': ids}))
print(f'{len(rows)} entities are reachable from the {len(ids)} retrieved chunks:\n')
for r in rows[:14]:
    print(f"  {r['chunk']:<14} {r['label']:<18} {r['entity'][:34]:<36} matched on {r['matched_text']!r}")
print(f'  ... and {max(0, len(rows)-14)} more')

32 entities are reachable from the 4 retrieved chunks:

  chunk_p1_05    CourseBasket       Ability Enhancement Courses          matched on 'Ability Enhancement Courses'
  chunk_p1_05    CourseBasket       Discipline Core                      matched on 'Discipline Core'
  chunk_p1_05    CourseBasket       Discipline Elective                  matched on 'Discipline Elective'
  chunk_p1_05    CourseBasket       Discipline Linked Engineering Cour   matched on 'Discipline Linked Engineering Courses'
  chunk_p1_05    CourseBasket       Foundation Core                      matched on 'Foundation Core'
  chunk_p1_05    CourseBasket       Open Elective                        matched on 'Open Elective'
  chunk_p1_05    CourseBasket       Project and Internship               matched on 'Project and Internship'
  chunk_p1_05    CourseBasket       Skill Enhancement Courses            matched on 'Skill Enhancement Courses'
  chunk_p1_05    CourseBasket       Specialization Elective              ma

## Stage 3 — Graph expansion: facts that cross chunk boundaries

From those entities, take one hop. This is what recovers the facilities: in the *text* they sit
in a different chunk from the school, but in the *graph* `HAS_FACILITY` is a single edge away.

> **Chunk boundaries are an artifact of how the text was split. Graph edges are not.**

Two details in the Cypher that are easy to get wrong, and both were wrong in my first version:

- **Direction.** The traversal is undirected so we reach neighbours on both sides, but each
  emitted fact must keep the edge's true direction, read from `startNode`/`endNode`. Reading it
  off the traversal produced reversed nonsense like `10+2 marks -REQUIRES-> B.Des`.
- **Edge properties.** `evidence`, `condition_group`, `source_note` live on the *edges*. Emitting
  bare subject-predicate-object silently discards the curated nuggets that justify having a graph.

In [4]:
print(H.NEIGHBOURHOOD.strip())

MATCH (e)-[:MENTIONED_IN]->(c:Chunk) WHERE c.id IN $ids OR any(l IN $labels WHERE l IN labels(e))
MATCH (e)-[r]-(n)
WHERE NOT n:Chunk AND NOT n:Document AND type(r) <> 'MENTIONED_IN'
WITH DISTINCT r, startNode(r) AS s, endNode(r) AS o
RETURN labels(s)[0] AS subject_label,
       coalesce(s.name, s.version, toString(s.number)) AS subject,
       type(r) AS predicate,
       coalesce(o.name, o.version, toString(o.number)) AS object,
       apoc.map.removeKeys(properties(r), ['demo']) AS props
ORDER BY subject_label, subject, predicate, object


In [5]:
with driver.session(database=database) as s:
    facts = [H.format_fact(r) for r in s.run(H.NEIGHBOURHOOD, {'ids': ids, 'labels': []})]
print(f'{len(facts)} facts from 1-hop expansion. The ones that answer the question:\n')
for f in facts:
    if 'HAS_FACILITY' in f or 'OFFERS' in f:
        print('   ', f)
print('\nEdge properties survive - these are the curated facts, not just topology:\n')
for f in facts:
    if '[' in f and ('EXEMPT_FROM' in f or 'condition_group' in f):
        print('   ', textwrap.shorten(f, 118))

54 facts from 1-hop expansion. The ones that answer the question:

    VIT School of Design -HAS_FACILITY-> 3D-iD Studio
    VIT School of Design -HAS_FACILITY-> Ergonomics Lab
    VIT School of Design -HAS_FACILITY-> PROTICS Studio
    VIT School of Design -HAS_FACILITY-> Painting Booth
    VIT School of Design -HAS_FACILITY-> Smart PD Lab
    VIT School of Design -OFFERS-> B.Des
    VIT School of Design -OFFERS-> M.Des

Edge properties survive - these are the curated facts, not just topology:

    B.Des -ADMITS_VIA [condition_group=OR_1]-> UCEED
    B.Des -ADMITS_VIA [condition_group=OR_1]-> V-DAT
    B.Des -REQUIRES [condition_group=AND; mandatory=True; note=in the qualifying examination]-> 10+2 marks
    B.Des -REQUIRES [condition_group=OR_1; mandatory=True]-> UCEED score
    B.Des -REQUIRES [condition_group=OR_1; mandatory=True]-> V-DAT score
    B.Tech CSE -REQUIRES [condition_group=AND; mandatory=True; note=counselling to choose the preferred campus]-> [...]
    B.Tech Civil -RE

## Stage 4 — Enter the graph from the question, not only from the chunks

Stages 2–3 seed the graph **from retrieved chunks**. That inherits every retrieval failure: ask
*"which schools are named?"*, have retrieval return page-1 chunks that mention no school, and the
graph is **never asked about schools at all**.

> A hybrid that enters the graph only through retrieved chunks is not a hybrid. It is vector RAG
> with extra steps, and it fails exactly where vector RAG fails.

So the question is also parsed for **type words** — "schools" → `:School`, "facilities" →
`:Facility`. This is deliberately simple (a word list, not an LLM) so students can see there is
no magic. A production system would use an LLM or a learned linker; the principle is identical.

Note the last example: a pure prose question yields no types, so no graph noise is added.

In [6]:
for q in ['Which schools are named in the document?',
          'List all the facilities at the school that offers B.Des.',
          'How many distinct entrance examinations are named?',
          'How many Academic Council meetings are referenced?',
          'What do employers expect from students?']:
    print(f'  {q[:56]:<58} -> {H.labels_from_question(q) or "(none - pure prose)"}')

  Which schools are named in the document?                   -> ['School']
  List all the facilities at the school that offers B.Des.   -> ['School', 'Facility']
  How many distinct entrance examinations are named?         -> ['EntranceExam']
  How many Academic Council meetings are referenced?         -> ['CouncilMeeting']
  What do employers expect from students?                    -> (none - pure prose)


In [7]:
# Proof that stage 4 matters: the schools question, with and without it.
Q = 'Which schools are named in the document?'
with driver.session(database=database) as s:
    seeded = collection.query(query_embeddings=[embed_query(model, Q)],
                              n_results=4, include=['documents'])['ids'][0]
    without = [H.format_fact(r) for r in s.run(H.NEIGHBOURHOOD, {'ids': seeded, 'labels': []})]
    withq   = [H.format_fact(r) for r in s.run(H.NEIGHBOURHOOD,
                                               {'ids': seeded, 'labels': H.labels_from_question(Q)})]
print(f'retrieved chunks: {seeded}')
print(f'  chunk-seeded only     : {len(without):>3} facts, mentions a School? '
      f'{any("School" in f for f in without)}')
print(f'  + question type words : {len(withq):>3} facts, mentions a School? '
      f'{any("School" in f for f in withq)}')
print('\nWithout stage 4 the graph contributes nothing about schools, and hybrid')
print('reproduces naive RAG\'s failure exactly.')

retrieved chunks: ['chunk_p1_01', 'chunk_p1_02', 'chunk_p1_05', 'chunk_p2_03']
  chunk-seeded only     :  63 facts, mentions a School? False
  + question type words :  71 facts, mentions a School? True

Without stage 4 the graph contributes nothing about schools, and hybrid
reproduces naive RAG's failure exactly.


## Stage 5 — The census: complete sets

The block that repairs counting. For every relevant entity type, the graph contributes **every**
member — not just those that happened to be retrieved.

```cypher
MATCH (n:EntranceExam) RETURN count(n), collect(n.name)   // 4. The set, not a sample.
```

This is the one thing top-k retrieval can never supply, at any *k*, because *k* is chosen before
anyone knows how much evidence the question needs.

`Feature` and `Chunk` are excluded from the census — dumping eleven long feature texts would
drown the prompt. Choosing what to enumerate is a design decision, not an automatic one.

In [8]:
print('Types eligible for a census:', ', '.join(sorted(H.CENSUS_LABELS)))
print()
with driver.session(database=database) as s:
    for label in ['EntranceExam', 'School', 'Facility']:
        row = s.run(H.COMPLETE_SET, {'label': label, 'demo': 'ffcs_kg'}).single()
        print(f"  ALL {label} ({row['n']}): {', '.join(sorted(row['members']))}")

Types eligible for a census: AdmissionCriterion, CouncilMeeting, CourseBasket, EntranceExam, Facility, GoverningBody, Programme, Regulation, School

  ALL EntranceExam (4): UCEED, V-DAT, VITEEE, VITMEE
  ALL School (2): VIT Business School, VIT School of Design
  ALL Facility (5): 3D-iD Studio, Ergonomics Lab, PROTICS Studio, Painting Booth, Smart PD Lab


## The assembled prompt

Everything above, concatenated. Read it as the model reads it — this is the single most useful
cell to project in a class, because it makes "context engineering" concrete.

Note the instruction that ties the blocks together: *when the question asks "how many" or "list
all", trust COMPLETE SETS over the passages, because the passages are only a sample.* Without
that line the model has two sources and no rule for preferring one.

In [9]:
with driver.session(database=database) as s:
    ids2, context, facts2, census2 = H.build_context(s, QUESTION, model, collection)
print(f'{len(ids2)} passages | {len(facts2)} graph facts | {len(census2)} complete sets')
print(f'total context: {len(context)} characters\n')
print(H.PROMPT.format(context=context, question=QUESTION)[:2600])
print('\n... [truncated for display]')

4 passages | 54 graph facts | 7 complete sets
total context: 6824 characters

Answer the question using the evidence below.

You are given three kinds of evidence:
- PASSAGES: verbatim text from the document. Use these for wording, rationale and nuance.
- GRAPH FACTS: curated relationships extracted from the whole document.
- COMPLETE SETS: exhaustive lists. When a question asks "how many" or "list all",
  trust COMPLETE SETS over the passages, because the passages are only a sample of
  the document while the complete sets cover all of it.

Rules:
1. Be concise (1-4 sentences).
2. Cite passages you quote using their tag, e.g. [chunk_p3_02].
3. Then, under a line "SOURCES:", one line per passage used:
   chunk_id | verbatim sentence copied exactly from that passage
4. If you relied on a complete set or a graph fact rather than a passage, write
   "GRAPH | <the fact>" as a source line instead.
5. If the evidence does not answer the question, say so explicitly.

PASSAGES:
[chunk_p3_06] (

## Evaluating without spending API quota: context recall

An answer can only be right if the evidence reached the prompt. So before asking whether the
model answered correctly, ask a cheaper and far more diagnostic question:

> **Did the assembled context actually contain what the answer needs?**

This runs instantly, costs nothing, and separates two failures that look identical in the output:
*the evidence never arrived* versus *the evidence arrived and the model misread it*. If context
recall is 100% and the answer is still wrong, the fix is in the prompt or the model — not the
retriever.

Each requirement accepts **alternative surface forms**, because the same fact is written two ways:
the text says *"There is NO Entrance Examination"*, the graph says `EXEMPT_FROM`. Testing only for
the sentence would score the graph as having lost information it actually holds.

In [10]:
with driver.session(database=database) as s:
    rows = ablation.run(s, model=model, collection=collection)

print(f"{'id':<5} {'kind':<7} {'need':<5} {'passages':<10} {'graph':<10} {'hybrid':<9}")
print('-' * 56)
for r in rows:
    n = r['required']
    print(f"{r['id']:<5} {r['kind']:<7} {n:<5} "
          f"{str(r['passages'])+'/'+str(n):<10} {str(r['graph'])+'/'+str(n):<10} "
          f"{str(r['hybrid'])+'/'+str(n):<9}")
print()
for v in ablation.VARIANTS:
    got, need = sum(r[v] for r in rows), sum(r['required'] for r in rows)
    size = sum(r[v + '_chars'] for r in rows) // len(rows)
    print(f"  {v:<9} context recall {got:>2}/{need} = {got/need:>4.0%}   avg context {size:>5} chars")

id    kind    need  passages   graph      hybrid   
--------------------------------------------------------
C1    local   1     1/1        1/1        1/1      
C2    local   1     1/1        1/1        1/1      
C13   global  4     2/4        4/4        4/4      
C15   global  2     0/2        2/2        2/2      
C7    global  5     0/5        5/5        5/5      
C9    global  10    9/10       10/10      10/10    
K1    prose   2     2/2        0/2        2/2      
K2    prose   1     1/1        0/1        1/1      
K3    prose   1     1/1        0/1        1/1      

  passages  context recall 17/27 =  63%   avg context  2337 chars
  graph     context recall 23/27 =  85%   avg context  3677 chars
  hybrid    context recall 27/27 = 100%   avg context  6017 chars


### Reading the table

- **passages 63%** — perfect on local and prose questions, poor on global ones. It never sees
  enough of the document to count anything.
- **graph 85%** — perfect on every global question, and **zero on all three prose questions**.
  The rationale, the objectives, the "holistic viewpoint" were never modelled, so no query can
  reach them.
- **hybrid 100%** — but at **2.5× the context size**. That is the real trade, and it is not free:
  more tokens per call, more cost, more latency, and more room for the model to be distracted.

The complementarity is exact. Neither column is a subset of the other, which is why the union
is worth building.

## Measured end-to-end answers

Recorded from live runs. Two groups: the questions naive RAG got wrong, and the questions the
graph has no data for.

In [11]:
print('A) Questions NAIVE RAG got wrong - global aggregates\n')
for qid, r in hybrid_agg.items():
    print('=' * 98)
    print(f"[{qid}] {r['question']}")
    print(f"  truth  : {r['gold']}")
    print(f"  naive  : {naive[qid]['answer'][:130]}")
    print(f"  hybrid : {r['answer'][:250]}")
    print(f"  context: {r['n_facts']} facts, {r['n_census']} complete sets\n")

A) Questions NAIVE RAG got wrong - global aggregates

[C13] How many distinct entrance examinations are named in the document? List them.
  truth  : 4: VITEEE, VITMEE, UCEED, V-DAT
  naive  : There are 2 distinct entrance examinations named in the document: VITEEE, VITMEE.
  hybrid : There are 4 distinct entrance examinations named in the document: UCEED, V-DAT, VITEEE, and VITMEE. Dates for exams like VITEEE and VITMEE are announced through the university website or media [chunk_p3_02].
  context: 63 facts, 5 complete sets

[C15] Which schools are named in the document?
  truth  : VIT School of Design (V-SIGN) and VIT Business School
  naive  : The context does not contain the answer.
  hybrid : The document names two schools: VIT Business School and VIT School of Design.
  context: 71 facts, 5 complete sets

[C7] List all the facilities at the school that offers B.Des.
  truth  : PROTICS Studio, 3D-iD Studio, Smart PD Lab, Ergonomics Lab, Painting Booth
  naive  : The facilities at t

In [12]:
print('B) Questions the KNOWLEDGE GRAPH has no data for - unmodelled prose\n')
for qid in ['K1', 'K2', 'K3', 'K4', 'K5', 'K7']:
    r = hybrid_gaps[qid]
    print('=' * 98)
    print(f"[{qid}] {r['question']}")
    print(f"  truth  : {r['gold']}")
    print(f"  KAG    : nothing in the graph to query")
    print(f"  hybrid : {r['answer'][:200]}")
    for c in r['citations'][:2]:
        print(f"           [{c['citation']}] \"{c['quote'][:66]}\"")
    print()

B) Questions the KNOWLEDGE GRAPH has no data for - unmodelled prose

[K1] What do employers expect from students?
  truth  : Multi-disciplinary competency, leadership skills, ICT ready.
  KAG    : nothing in the graph to query
  hybrid : Employers expect students to have multi-disciplinary competency, leadership skills, and be Information and Communication Technology (ICT) ready [chunk_p1_03].
           [page 1, passage 3 of 5] "Employers expect students to have multi-disciplinary competency, l"

[K2] What is the main objective of the design programmes at V-SIGN?
  truth  : Create a new breed of problem solvers in industrial design.
  KAG    : nothing in the graph to query
  hybrid : The main objective of the V-SIGN programmes is "to create a new breed of problem solvers in the domain of industrial design" [chunk_p3_04]. Additionally, the programmes aim to develop students' skills
           [page 3, passage 4 of 6 (paragraph 2)] "The main objective of the programmes is to create a ne

## Four bugs, kept in on purpose

Every one of these produced a *plausible* wrong answer rather than an error. That is the danger
of this class of system: it fails quietly.

**1. Silent truncation.** The fact list was capped at `facts[:60]`. Facts were sorted
alphabetically by label, so `School -HAS_FACILITY-> PROTICS Studio` sorted near the end and was
cut. The context looked full and the answer looked reasonable.
→ *Never truncate an ordered list at an arbitrary boundary. Rank by relevance, or keep it all.*

**2. Seeding only from retrieved chunks.** Graph expansion started from entities in the retrieved
passages, so a retrieval miss became a hybrid miss. Fixed by stage 4.
→ *A hybrid must be able to enter the graph even when retrieval looked in the wrong place.*

**3. Undirected traversal emitting directed facts.** `MATCH (e)-[r]-(n)` reaches neighbours on
both sides; formatting the result as `e -type-> n` produced reversed facts such as
`10+2 marks -REQUIRES-> B.Des`. An LLM has no way to know that is backwards.
→ *Read direction from `startNode`/`endNode`, never from traversal order.*

**4. The same formatting logic written twice.** The ablation module reformatted facts
independently and omitted edge properties, so the graph scored 0 on a question whose answer it
actually contained — a measurement bug that would have led to a wrong conclusion about the graph.
→ *Define the serialisation once and import it. Divergent copies produce confident bad data.*

Bug 4 is the one worth dwelling on. Bugs 1–3 degraded the system. Bug 4 corrupted the
**evaluation**, which is worse: it would have been reported as a finding.

## Cost, tuning, and when hybrid still loses

### The knobs

| Knob | Effect | Failure mode if wrong |
|---|---|---|
| `k` (chunks retrieved) | more prose evidence | bigger prompt; more distraction |
| hops in expansion | more structural context | 2 hops on this graph returns nearly everything |
| `CENSUS_LABELS` | which types get enumerated | enumerate `Feature` and the prompt drowns |
| `LABEL_WORDS` | how the question reaches the graph | a missing synonym silently disables stage 4 |

`LABEL_WORDS` deserves suspicion: it is a hand-written word list. Ask *"which departments are
named?"* instead of *"which schools"* and stage 4 contributes nothing, with no error raised. A
production system should use an LLM or a trained linker here.

### Where hybrid does not help

- **The document is wrong.** For M.Tech the source says "VITEEE ranks" where VITMEE is meant.
  Hybrid sees both the erroneous sentence and the corrected graph fact, and may follow either.
  Only curation fixes this, and curation is human work.
- **The fact is genuinely absent.** The M.Des admission route is missing because the PDF is
  truncated. Hybrid correctly reports it as unstated — no retrieval strategy invents facts
  that were never written.
- **The ontology is wrong.** If the graph models something incorrectly, hybrid now has a
  confident wrong fact *and* the text it contradicts. More context, more confusion.
- **Cost.** 2.5× the prompt for every query, plus a graph to build and maintain. On a corpus
  where nobody asks aggregate questions, that is pure overhead.

> Hybrid is not "RAG plus a graph, therefore better". It is a bet that your users ask both local
> and global questions. Check that assumption before paying for it.

## Exercises

1. **Break stage 4.** Ask *"which departments are named in the document?"* Confirm
   `labels_from_question` returns `[]` and that hybrid then fails exactly like naive RAG. Add a
   synonym and watch it recover.
2. **Restore bug 1.** Put back `facts[:60]` in `build_context` and re-run the facilities question.
   Note that the answer is fluent and wrong, and that context recall drops while nothing errors.
3. **Measure the k trade-off.** Run the ablation at `k=2, 4, 8`. At what point does `passages`
   recall catch up with `hybrid`, and how large is the prompt by then?
4. **Add a census type.** Put `Feature` in `CENSUS_LABELS`, re-run, and judge whether the
   improvement on "list all features" is worth the prompt size.
5. **Write the missing edge.** Pick one prose question the graph cannot answer (K1–K5, K7), extend
   the ontology to model it, reload, and confirm the graph column improves. Then find a new
   question it still cannot answer — this is the point of the exercise.

In [13]:
driver.close()
print('Done.')

Done.
